In [28]:
import pandas as pd
import numpy as np

# Show all columns when printing
pd.set_option('display.max_columns', None)

# Load your main dataset
df = pd.read_csv('../data/raw/environmental_impact.csv')
print("Main dataset shape:", df.shape)
print(df.head())

# Load ADEME emissions data
df_ademe = pd.read_csv(
    '../data/external/ademe_base_carbone.csv',
    sep=';',
    encoding='latin-1',
    low_memory=False
)
print("ADEME dataset shape:", df_ademe.shape)
print(df_ademe.head())

Main dataset shape: (3678, 18)
  route_name            origin           destination service_type  \
0     ICE 18      Hauptbahnhof  Berlin Gesundbrunnen          day   
1      IC 51      Hauptbahnhof          Dortmund Hbf          day   
2     ICE 28  Hamburg Altona S          Hauptbahnhof          day   
3      IC 51      Hauptbahnhof              Koln Hbf          day   
4     ICE 11       Munchen Hbf          Hauptbahnhof          day   

        route_name_simple origin_country destination_country  distance_km  \
0  → Berlin Gesundbrunnen             DE                  DE       238.64   
1              → Dortmund             DE                  DE       256.33   
2      Hamburg Altona S →             DE                  DE       296.60   
3                  → Koln             DE                  DE       285.66   
4               Munchen →             DE                  DE       317.10   

  operator type  train_gco2_pkm  plane_gco2_pkm  train_co2_kg  plane_co2_kg  \
0       DB  

In [16]:
# -----------------------------
#  CLEAN ADEME DATA 
# -----------------------------
# Convert French format numbers (e.g., "0,172") → float
df_ademe['Total poste non décomposé'] = (
    df_ademe['Total poste non décomposé']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .str.replace(' ', '', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)


In [17]:
# -----------------------------
#  EXTRACT TRAIN EMISSIONS FROM ADEME
# -----------------------------
train_ademe = df_ademe[
    (df_ademe['Unité français'] == 'kgCO2e/passager.km') &
    (df_ademe['Nom base français'].str.contains(
        'TGV|TER|Intercités|Train',
        case=False, na=False
    )) &
    (df_ademe['Total poste non décomposé'] > 0)
]

# Take average value (simple + robust)
train_co2_factor = train_ademe['Total poste non décomposé'].mean()

print("Train CO2 (kg/km):", train_co2_factor)


Train CO2 (kg/km): 0.04899697674418605


In [18]:
# -----------------------------
# EXTRACT PLANE EMISSIONS FROM ADEME
# -----------------------------
plane_ademe = df_ademe[
    (df_ademe['Unité français'] == 'kgCO2e/passager.km') &
    (df_ademe['Nom base français'].str.contains(
        'Avion|aérien|vol',
        case=False, na=False
    )) &
    (df_ademe['Total poste non décomposé'] > 0)
]


In [20]:
# -----------------------------
# 5. CLEAN MAIN DATASET
# -----------------------------
# Remove rows without distance
df = df.dropna(subset=['distance_km'])

# Standardize operator names
df['operator'] = df['operator'].replace({
    'SNCF VOYAGEURS': 'SNCF'
})

print("After cleaning:", df.shape)



After cleaning: (3487, 21)


In [21]:
# -----------------------------
# 6. COMPUTE EMISSIONS USING ADEME
# -----------------------------

# Train = SAME value from ADEME
df['train_gco2_pkm'] = train_co2_factor

# Plane depends on distance
def get_plane_co2(distance):
    if distance < 1000:
        return plane_short
    elif distance < 2000:
        return plane_medium
    else:
        return plane_long

df['plane_gco2_pkm'] = df['distance_km'].apply(get_plane_co2)

# IMPORTANT: ADEME already in kg → no /1000
df['train_co2_kg'] = df['distance_km'] * df['train_gco2_pkm']
df['plane_co2_kg'] = df['distance_km'] * df['plane_gco2_pkm']

# Target variable
df['co2_savings_kg'] = df['plane_co2_kg'] - df['train_co2_kg']

# Ratio (optional)
df['co2_ratio'] = df['plane_co2_kg'] / df['train_co2_kg']


In [22]:
# -----------------------------
# 7. REMOVE DUPLICATES
# -----------------------------
df = df.drop_duplicates(subset=['origin', 'destination', 'type'])


In [23]:
# -----------------------------
# 8. FEATURE ENGINEERING (ML)
# -----------------------------
# International route
df['is_international'] = (
    df['origin_country'] != df['destination_country']
).astype(int)

# Distance category
df['distance_category'] = pd.cut(
    df['distance_km'],
    bins=[0, 500, 1000, 2000, 10000],
    labels=['short', 'medium', 'long', 'very_long']
)



In [25]:

# -----------------------------
# 9. CREATE ML DATASET
# -----------------------------
features = [
    'distance_km',
    'service_type',
    'operator',
    'origin_country',
    'destination_country',
    'type',
    'is_international',
    'distance_category'
]

target = 'co2_savings_kg'

# Convert categorical → numerical
df_ml = pd.get_dummies(df[features], drop_first=True)

# Add target
df_ml[target] = df[target]

# Final cleanup
df_ml = df_ml.fillna(0)


# -----------------------------
# 10. SAVE FILES
# -----------------------------
df.to_csv('../data/processed/routes_processed.csv', index=False)
df_ml.to_csv('../data/processed/ml_dataset.csv', index=False)

print("✅ DONE")
print("Final ML shape:", df_ml.shape)

✅ DONE
Final ML shape: (3487, 84)


In [27]:
df_ml.head()

,distance_km,is_international,service_type_night,operator_ASM-snb (Aare Seeland mobil (snb)),operator_AVA-bd (Aargau Verkehr AG),operator_AVA-wsb (Aargau Verkehr AG),operator_BLS-bls (BLS AG (bls)),operator_BOB (Berner Oberland-Bahnen),operator_CD,operator_CFL,operator_CH,operator_CJ (Chemins de fer du Jura),operator_Conseil Régional Auvergne - Rhône-Alpes,operator_DB,operator_DB Fernverkehr (Codesharing),operator_DB Regio (DB Regio AG Baden-Württemberg),operator_DFB (Dampfbahn Furka-Bergstrecke),operator_DPN,operator_DSB,operator_Dänische Staatsbahnen,operator_EC,operator_Eurostar,operator_FART (Ferrovie Autolinee Regionali Ticinesi),operator_FS,operator_LEB (Lausanne-Echallens-Bercher),operator_MAV,operator_MBC (Transports de la région Morges-Bière-Cossonay),operator_MGB-bvz (Matterhorn Gotthard Bahn (bvz)),operator_MGB-fo (Matterhorn Gotthard Bahn (fo)),operator_MOB (Montreux-Oberland Bernois),operator_NS,operator_NStCM (Nyon-St-Cergue-Morez),operator_OCEdefault,operator_PKP,operator_RA (Regionalps),operator_RBS (Regionalverkehr Bern-Solothurn),operator_RE,operator_RZD,operator_RhB (Rhätische Bahn),operator_SBB,operator_SBB (SBB GmbH),operator_SBB (Schweizerische Bundesbahnen SBB),operator_SBB GmbH (SBB GmbH (Grenzverkehr)),operator_SNCF,operator_SNCF (Société Nationale des Chemins de fer Français),operator_SNCF Voyageurs EA,operator_SNCF Voyageurs LO,operator_SNCF Voyageurs SA,operator_SOB-sob (Schweizerische Südostbahn (sob)),operator_SWX,operator_SZU (Sihltal-Zürich-Uetliberg-Bahn),operator_TER,operator_THURBO (THURBO),operator_TMR-mc (Transports de Martigny et Régions (mc)),operator_TPC (Transports Publics du Chablais),operator_TPF (Transports publics fribourgeois),operator_TRAVYS (Transports Vallée de Joux-Yverdon-Ste-Croix),operator_TRN-cmn (Transports Publics Neuchâtelois SA (cmn)),operator_TRN-rvt (Transports Publics Neuchâtelois SA (rvt)),operator_Trenitalia,operator_Unknown,operator_VDBB (Verein Dampfbahn Bern),operator_VHB (FPLAN VHB SBP),operator_ZB (Zentralbahn),operator_ÖBB,operator_ÖBB (Österreichische Bundesbahnen),origin_country_BE,origin_country_CH,origin_country_DE,origin_country_FR,origin_country_GB,origin_country_IT,origin_country_NL,destination_country_CH,destination_country_DE,destination_country_FR,destination_country_GB,destination_country_IT,destination_country_NL,type_night,distance_category_medium,distance_category_long,distance_category_very_long,co2_savings_kg
0,238.64,0,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,24.342001
1,256.33,0,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,26.146435
2,296.60,0,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,30.254097
3,285.66,0,False,False,False,False,False,False,False

In [19]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

# -----------------------------
# 1. LOAD DATA
# -----------------------------
df = pd.read_csv('../data/raw/environmental_impact.csv')

df_ademe = pd.read_csv(
    '../data/external/ademe_base_carbone.csv',
    sep=';',
    encoding='latin-1',
    low_memory=False
)

print("Initial shape:", df.shape)


# -----------------------------
# 2. CLEAN ADEME DATA (IMPORTANT)
# -----------------------------
# Convert French format numbers (e.g. "0,172") → float
df_ademe['Total poste non décomposé'] = (
    df_ademe['Total poste non décomposé']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .str.replace(' ', '', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)


# -----------------------------
# 3. EXTRACT TRAIN EMISSIONS FROM ADEME
# -----------------------------
train_ademe = df_ademe[
    (df_ademe['Unité français'] == 'kgCO2e/passager.km') &
    (df_ademe['Nom base français'].str.contains(
        'TGV|TER|Intercités|Train',
        case=False, na=False
    )) &
    (df_ademe['Total poste non décomposé'] > 0)
]

# Take average value (simple + robust)
train_co2_factor = train_ademe['Total poste non décomposé'].mean()

print("Train CO2 (kg/km):", train_co2_factor)


# -----------------------------
# 4. EXTRACT PLANE EMISSIONS FROM ADEME
# -----------------------------
plane_ademe = df_ademe[
    (df_ademe['Unité français'] == 'kgCO2e/passager.km') &
    (df_ademe['Nom base français'].str.contains(
        'Avion|aérien|vol',
        case=False, na=False
    )) &
    (df_ademe['Total poste non décomposé'] > 0)
]

# Create distance-based values
plane_short = plane_ademe['Total poste non décomposé'].quantile(0.75)
plane_medium = plane_ademe['Total poste non décomposé'].median()
plane_long = plane_ademe['Total poste non décomposé'].quantile(0.25)

print("Plane short/medium/long:", plane_short, plane_medium, plane_long)


# -----------------------------
# 5. CLEAN MAIN DATASET
# -----------------------------
# Remove rows without distance
df = df.dropna(subset=['distance_km'])

# Standardize operator names
df['operator'] = df['operator'].replace({
    'SNCF VOYAGEURS': 'SNCF'
})

print("After cleaning:", df.shape)


# -----------------------------
# 6. COMPUTE EMISSIONS USING ADEME
# -----------------------------

# Train = SAME value from ADEME
df['train_gco2_pkm'] = train_co2_factor

# Plane depends on distance
def get_plane_co2(distance):
    if distance < 1000:
        return plane_short
    elif distance < 2000:
        return plane_medium
    else:
        return plane_long

df['plane_gco2_pkm'] = df['distance_km'].apply(get_plane_co2)

# IMPORTANT: ADEME already in kg → no /1000
df['train_co2_kg'] = df['distance_km'] * df['train_gco2_pkm']
df['plane_co2_kg'] = df['distance_km'] * df['plane_gco2_pkm']

# Target variable
df['co2_savings_kg'] = df['plane_co2_kg'] - df['train_co2_kg']

# Ratio (optional)
df['co2_ratio'] = df['plane_co2_kg'] / df['train_co2_kg']


# -----------------------------
# 7. REMOVE DUPLICATES
# -----------------------------
df = df.drop_duplicates(subset=['origin', 'destination', 'type'])


# -----------------------------
# 8. FEATURE ENGINEERING (ML)
# -----------------------------
# International route
df['is_international'] = (
    df['origin_country'] != df['destination_country']
).astype(int)

# Distance category
df['distance_category'] = pd.cut(
    df['distance_km'],
    bins=[0, 500, 1000, 2000, 10000],
    labels=['short', 'medium', 'long', 'very_long']
)


# -----------------------------
# 9. CREATE ML DATASET
# -----------------------------
features = [
    'distance_km',
    'service_type',
    'operator',
    'origin_country',
    'destination_country',
    'type',
    'is_international',
    'distance_category'
]

target = 'co2_savings_kg'

# Convert categorical → numerical
df_ml = pd.get_dummies(df[features], drop_first=True)

# Add target
df_ml[target] = df[target]

# Final cleanup
df_ml = df_ml.fillna(0)


# -----------------------------
# 10. SAVE FILES
# -----------------------------
df.to_csv('../data/processed/routes_processed.csv', index=False)
df_ml.to_csv('../data/processed/ml_dataset.csv', index=False)

print("✅ DONE")
print("Final ML shape:", df_ml.shape)

Initial shape: (3678, 18)
Train CO2 (kg/km): 0.04899697674418605
Plane short/medium/long: 0.151 0.0946 0.0211
After cleaning: (3492, 18)
✅ DONE
Final ML shape: (3487, 84)
